# Projet - Tache 1 : Collecte de donnees

Version simple :
- requete SPARQL avec `SPARQLWrapper`
- resultats dans un `DataFrame` pandas
- une petite fonction pour lire les metadonnees Commons
- telechargement des vignettes et export vers `data/images_metadata.json`


In [16]:
# A executer une fois si besoin
# !pip install SPARQLWrapper pandas Pillow requests

import html
import re
import time
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import requests
from PIL import Image
from PIL.ExifTags import TAGS
from SPARQLWrapper import JSON, SPARQLWrapper

BASE_DIR = Path('.')
IMAGES_DIR = BASE_DIR / 'images'
DATA_DIR = BASE_DIR / 'data'
IMAGES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

USER_AGENT = 'CPE-ML-Project/1.0 (academic project; contact: mathislambert.dev@gmail.com)'
REQUEST_DELAY_SECONDS = 0.5


In [17]:
query = '''
SELECT ?fleur ?fleurLabel ?image
       (GROUP_CONCAT(DISTINCT ?couleurLabel; separator=", ") AS ?couleurs)
WHERE {
  ?fleur wdt:P2827 ?couleur ;
         wdt:P18 ?image .
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "fr,en".
  }
}
GROUP BY ?fleur ?fleurLabel ?image
LIMIT 120
'''

sparql = SPARQLWrapper('https://query.wikidata.org/sparql', agent=USER_AGENT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()['results']['bindings']

df = pd.DataFrame([
    {
        'wikidata_id': r['fleur']['value'].split('/')[-1],
        'label': r.get('fleurLabel', {}).get('value', ''),
        'image_url': r['image']['value'],
        'colors': r.get('couleurs', {}).get('value', '')
    }
    for r in results
]).drop_duplicates(subset='image_url').reset_index(drop=True)

print('Nombre de lignes :', len(df))
df.head()


Nombre de lignes : 120


,wikidata_id,label,image_url,colors
0,Q26745,érable plane,http://commons.wikimedia.org/wiki/Special:File...,
1,Q26745,érable plane,http://commons.wikimedia.org/wiki/Special:File...,
2,Q26745,érable plane,http://commons.wikimedia.org/wiki/Special:File...,
3,Q27164,marguerite commune,http://commons.wikimedia.org/wiki/Special:File...,
4,Q27164,marguerite commune,http://commons.wikimedia.org/wiki/Special:File...,


In [18]:
def clean_html(value):
    if value is None:
        return None
    value = html.unescape(str(value))
    value = re.sub(r'<[^>]+>', ' ', value)
    return re.sub(r'\s+', ' ', value).strip() or None


def get_commons_metadata(image_url):
    commons_title = 'File:' + unquote(image_url.split('/Special:FilePath/')[1])
    response = requests.get(
        'https://commons.wikimedia.org/w/api.php',
        params={
            'action': 'query',
            'titles': commons_title,
            'prop': 'imageinfo',
            'iiprop': 'url|size|mime|extmetadata',
            'iiurlwidth': 800,
            'format': 'json'
        },
        headers={'User-Agent': USER_AGENT},
        timeout=60
    )
    info = next(iter(response.json()['query']['pages'].values()))['imageinfo'][0]
    ext = info.get('extmetadata', {})
    return {
        'commons_title': commons_title,
        'download_url': info.get('thumburl') or info.get('url'),
        'source_page_url': info.get('descriptionurl'),
        'source_file_url': info.get('url'),
        'license_short_name': clean_html(ext.get('LicenseShortName', {}).get('value')),
        'license_url': clean_html(ext.get('LicenseUrl', {}).get('value')),
        'artist': clean_html(ext.get('Artist', {}).get('value')),
        'credit': clean_html(ext.get('Credit', {}).get('value')),
        'date_time_original': clean_html(ext.get('DateTimeOriginal', {}).get('value')),
        'original_file_size_kb': round((info.get('size') or 0) / 1024, 2),
        'original_width': info.get('width'),
        'original_height': info.get('height')
    }


def get_exif(image_path):
    keep = {'Make', 'Model', 'DateTime', 'DateTimeOriginal'}
    with Image.open(image_path) as img:
        return {
            TAGS.get(tag_id, tag_id): str(value)
            for tag_id, value in img.getexif().items()
            if TAGS.get(tag_id, tag_id) in keep
        }


In [19]:
metadata = []

for i, row in df.head(100).iterrows():
    try:
        commons = get_commons_metadata(row['image_url'])
        extension = Path(commons['download_url']).suffix.lower() or '.jpg'
        file_name = f'image_{i+1:03d}{extension}'
        image_path = IMAGES_DIR / file_name

        if not image_path.exists():
            response = requests.get(commons['download_url'], headers={'User-Agent': USER_AGENT}, timeout=60)
            response.raise_for_status()
            image_path.write_bytes(response.content)

        with Image.open(image_path) as img:
            width, height = img.size
            image_format = img.format

        label = row['label'] if row['label'] and not row['label'].startswith('Q') else commons['commons_title'].replace('File:', '')

        metadata.append({
            'file_name': file_name,
            'wikidata_id': row['wikidata_id'],
            'label': label,
            'colors': row['colors'],
            'width': width,
            'height': height,
            'format': image_format,
            'file_size_kb': round(image_path.stat().st_size / 1024, 2),
            'source_url': row['image_url'],
            'source_page_url': commons['source_page_url'],
            'source_file_url': commons['source_file_url'],
            'license_short_name': commons['license_short_name'],
            'license_url': commons['license_url'],
            'artist': commons['artist'],
            'credit': commons['credit'],
            'date_time_original': commons['date_time_original'],
            'original_file_size_kb': commons['original_file_size_kb'],
            'original_width': commons['original_width'],
            'original_height': commons['original_height'],
            'exif': get_exif(image_path)
        })

        print(f'{len(metadata):03d} - {file_name}')
        time.sleep(REQUEST_DELAY_SECONDS)

    except Exception as e:
        print('Erreur :', row['image_url'], e)

metadata_df = pd.DataFrame(metadata)
metadata_df.head()


001 - image_001.jpg
002 - image_002.jpg
003 - image_003.jpg
004 - image_004.jpg
005 - image_005.jpg
006 - image_006.jpg
007 - image_007.jpg
008 - image_008.jpg
009 - image_009.jpg
010 - image_010.jpg
011 - image_011.jpg
012 - image_012.jpg
013 - image_013.jpg
014 - image_014.jpg
015 - image_015.jpg
016 - image_016.jpg
017 - image_017.jpg
018 - image_018.jpg
019 - image_019.jpg
020 - image_020.jpg
021 - image_021.jpg
022 - image_022.jpg
023 - image_023.jpg
024 - image_024.jpg
025 - image_025.jpg
026 - image_026.jpg
027 - image_027.jpg
028 - image_028.jpg
029 - image_029.jpg
030 - image_030.jpg
031 - image_031.jpg
032 - image_032.jpg
033 - image_033.jpg
034 - image_034.jpg
035 - image_035.jpg
036 - image_036.jpg
037 - image_037.jpg
038 - image_038.jpg
039 - image_039.jpg
040 - image_040.jpg
041 - image_041.jpg
042 - image_042.jpg
043 - image_043.jpg
044 - image_044.jpg
045 - image_045.jpg
046 - image_046.jpg
047 - image_047.jpg
048 - image_048.jpg
Erreur : http://commons.wikimedia.org/wi

,file_name,wikidata_id,label,colors,width,height,format,file_size_kb,source_url,source_page_url,source_file_url,license_short_name,license_url,artist,credit,date_time_original,original_file_size_kb,original_width,original_height,exif
0,image_001.jpg,Q26745,érable plane,,960,1280,JPEG,314.83,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:2020_y...,https://upload.wikimedia.org/wikipedia/commons...,CC BY-SA 4.0,https://creativecommons.org/licenses/by-sa/4.0,Dmitry Makeev,Own work,2020-05-24 15:25:36,11381.98,3900,3519,{}
1,image_002.jpg,Q26745,érable plane,,640,480,JPEG,59.47,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Ahornb...,https://upload.wikimedia.org/wikipedia/commons...,CC BY 3.0,https://creativecommons.org/licenses/by/3.0,Rabensteiner,Own work,"12 May 2008, 11:42:42 (according to Exif data)",2983.14,3008,2000,"{'Make': 'SONY', 'Model': 'CYBERSHOT    ', 'Da..."
2,image_003.jpg,Q26745,érable plane,,960,671,JPEG,195.20,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Spitz-...,https://upload.wikimedia.org/wikipedia/commons...,CC BY-SA 2.5,https://creativecommons.org/licenses/by-sa/2.5,Martin Bobka (= Martin120 ),Own work,2006-05-25,4177.63,2413,1686,{}
3,image_004.jpg,Q27164,marguerite commune,,960,720,JPEG,153.96,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Leucan...,https://upload.wikimedia.org/wikipedia/commons...,Public domain,NaN,Victor M. Vicente Selvas,Own work,2007-06-15,2320.18,2560,1920,{}
4,image_005.jpg,Q27164,marguerite commune,,960,720,JPEG,113.91,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Magerw...,https://upload.wikimedia.org/wikipedia/commons...,CC BY-SA 4.0,https://creativecommons.org/licenses/by-sa/4.0,Lupus in Saxonia,Own work,2016-05-17 13:28:19,5697.14,4252,3189,{}


In [20]:
output_path = DATA_DIR / 'images_metadata.json'
metadata_df.to_json(output_path, orient='records', force_ascii=False, indent=2)

print('JSON sauvegarde dans :', output_path)
print('Nombre final :', len(metadata_df))
metadata_df.head()


JSON sauvegarde dans : data/images_metadata.json
Nombre final : 52


,file_name,wikidata_id,label,colors,width,height,format,file_size_kb,source_url,source_page_url,source_file_url,license_short_name,license_url,artist,credit,date_time_original,original_file_size_kb,original_width,original_height,exif
0,image_001.jpg,Q26745,érable plane,,960,1280,JPEG,314.83,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:2020_y...,https://upload.wikimedia.org/wikipedia/commons...,CC BY-SA 4.0,https://creativecommons.org/licenses/by-sa/4.0,Dmitry Makeev,Own work,2020-05-24 15:25:36,11381.98,3900,3519,{}
1,image_002.jpg,Q26745,érable plane,,640,480,JPEG,59.47,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Ahornb...,https://upload.wikimedia.org/wikipedia/commons...,CC BY 3.0,https://creativecommons.org/licenses/by/3.0,Rabensteiner,Own work,"12 May 2008, 11:42:42 (according to Exif data)",2983.14,3008,2000,"{'Make': 'SONY', 'Model': 'CYBERSHOT    ', 'Da..."
2,image_003.jpg,Q26745,érable plane,,960,671,JPEG,195.20,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Spitz-...,https://upload.wikimedia.org/wikipedia/commons...,CC BY-SA 2.5,https://creativecommons.org/licenses/by-sa/2.5,Martin Bobka (= Martin120 ),Own work,2006-05-25,4177.63,2413,1686,{}
3,image_004.jpg,Q27164,marguerite commune,,960,720,JPEG,153.96,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Leucan...,https://upload.wikimedia.org/wikipedia/commons...,Public domain,NaN,Victor M. Vicente Selvas,Own work,2007-06-15,2320.18,2560,1920,{}
4,image_005.jpg,Q27164,marguerite commune,,960,720,JPEG,113.91,http://commons.wikimedia.org/wiki/Special:File...,https://commons.wikimedia.org/wiki/File:Magerw...,https://upload.wikimedia.org/wikipedia/commons...,CC BY-SA 4.0,https://creativecommons.org/licenses/by-sa/4.0,Lupus in Saxonia,Own work,2016-05-17 13:28:19,5697.14,4252,3189,{}
